# Image Region Selection for xradio Images

This notebook demonstrates the extended CRTF selection features available when working with
xradio-format sky images — five-dimensional DataArrays with dims
`(time, frequency, polarization, l, m)` and the corresponding named coordinates.

For basic selection on generic 2-D arrays using pixel coordinates, see
**`generic_image_selection.ipynb`** in the same directory.

## Features covered

| Section | Feature | Coordinate required |
|---|---|---|
| [lm-mode shapes](#lm-mode-shapes) | Angular offsets in arcsec / arcmin | `l`, `m` |
| world-mode shapes (coming soon) | Absolute RA / Dec | `right_ascension`, `declination` |
| `range=` (coming soon) | Frequency, velocity, or channel | `frequency` / `velocity` |
| `corr=` (coming soon) | Polarization (Stokes) | `polarization` |
| `time=` (coming soon) | Time range (MJD / ISO) | `time` |

In [ ]:
import math
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from astroviper.distributed.image_analysis.selection import select_mask
from astroviper.utils.plotting import generate_plot

## Building a representative sky DataArray

xradio sky images have five dimensions: `time`, `frequency`, `polarization`, `l`, `m`.
The spatial dimensions `l` and `m` are angular offsets (in **radians**) east and north
of the image reference direction.  In practice the values are tiny numbers — a
typical 100-arcsec-wide image spans roughly ±2.4 × 10⁻⁴ rad in each axis.

The helper below builds a small but realistic-looking sky DataArray that we will use
throughout this notebook.

In [ ]:
def make_sky(
    n_l: int = 100,
    n_m: int = 100,
    cell_arcsec: float = 1.0,
) -> xr.DataArray:
    """Build a synthetic xradio-like sky DataArray.

    The image contains a bright Gaussian blob near the centre so that
    the selected regions are easy to see.

    Parameters
    ----------
    n_l, n_m : int
        Number of pixels along l and m.
    cell_arcsec : float
        Pixel size in arcseconds.

    Returns
    -------
    xr.DataArray
        Shape (1, 1, 1, n_l, n_m) with dims (time, frequency, polarization, l, m).
    """
    arcsec = math.pi / (180 * 3600)          # 1 arcsec in radians
    cell = cell_arcsec * arcsec

    # l/m coords: centred on zero, spacing = cell_arcsec
    l_rad = (np.arange(n_l) - (n_l - 1) / 2.0) * cell
    m_rad = (np.arange(n_m) - (n_m - 1) / 2.0) * cell

    # A simple Gaussian blob: peak = 1.0, sigma = 10 arcsec
    ll, mm = np.meshgrid(l_rad, m_rad, indexing="ij")
    sigma = 10 * arcsec
    blob = np.exp(-(ll**2 + mm**2) / (2 * sigma**2))

    data = blob[np.newaxis, np.newaxis, np.newaxis, :, :]

    return xr.DataArray(
        data,
        dims=["time", "frequency", "polarization", "l", "m"],
        coords={
            "time":        (["time"],        [0.0]),
            "frequency":   (["frequency"],   [1.4e9]),
            "polarization":(["polarization"],["I"]),
            "l":           (["l"],           l_rad),
            "m":           (["m"],           m_rad),
        },
    )

sky = make_sky()
sky

The `l` and `m` coordinates carry the physical angular scale.  We will always express
region boundaries in **arcsec** or **arcmin**, and `select_mask` converts them to radians
internally before comparing against the coordinate grids.

The visualisation helper below calls `generate_plot` from `astroviper.utils.plotting`
for each figure — this ensures correct axis-label conventions, the right (x, y) storage
order, and proper colorbar handling.  Two figures are produced for each example:

1. **Sky image** — the intensity map with angular offset axes in arcseconds.
2. **Selected region** — the same image with the mask overlaid: excluded pixels are
   dimmed and the region boundary is drawn in white.

Following the standard astronomical convention for direction-cosine coordinates,
the **l-axis (east) increases to the LEFT** on both figures.

In [ ]:
def plot_sky_with_mask(sky: xr.DataArray, mask: xr.DataArray, title: str = "") -> None:
    """Show a sky image and a mask overlay using generate_plot from astroviper.utils.plotting.

    Two figures are produced:
      1. The plain sky image with properly labelled angular axes.
      2. The same image with the selected region indicated by a white boundary
         and excluded pixels dimmed.

    Parameters
    ----------
    sky : xr.DataArray
        5-D sky DataArray with dims (time, frequency, polarization, l, m).
        The l and m coordinates must be in radians.
    mask : xr.DataArray
        Boolean mask from select_mask(), aligned with *sky*.
    title : str
        Short description of the region shown in the second plot title.

    Notes
    -----
    * l and m are converted from radians to arcseconds for display.
    * The l-axis (east) is inverted so that east is to the LEFT — the standard
      astronomical orientation for images in direction-cosine coordinates.
    * generate_plot handles axis labelling, colorbar, and the correct (x, y)
      storage-order convention automatically via show_world_axes=True.
    """
    arcsec_rad = math.pi / (180 * 3600)

    # Squeeze the 5-D arrays to 2-D (l, m) for plotting
    sky_2d  = sky.isel(time=0, frequency=0, polarization=0)
    mask_2d = mask.squeeze()   # drops all size-1 dims -> (l, m)

    # Angular axes in arcseconds
    l_arcsec = sky_2d.coords["l"].values / arcsec_rad
    m_arcsec = sky_2d.coords["m"].values / arcsec_rad

    _kw = dict(
        show_world_axes=True,
        x_coords=l_arcsec,
        y_coords=m_arcsec,
        cmap="viridis",
        figsize=(5.5, 5.0),
    )

    # --- Panel 1: plain sky image ---
    fig, ax = generate_plot(sky_2d, title="Sky image", **_kw)
    ax.invert_xaxis()   # east (increasing l) is to the LEFT
    ax.set_xlabel("l offset  [arcsec]")
    ax.set_ylabel("m offset  [arcsec]")
    plt.tight_layout()
    plt.show()

    # --- Panel 2: sky image with mask overlay ---
    panel_title = f"Selected region — {title}" if title else "Selected region"
    fig, ax = generate_plot(sky_2d, title=panel_title, **_kw)
    ax.invert_xaxis()
    ax.set_xlabel("l offset  [arcsec]")
    ax.set_ylabel("m offset  [arcsec]")

    mask_np = np.asarray(mask_2d.values, dtype=float)
    # Dim excluded pixels with a semi-transparent grey overlay
    excluded = np.where(mask_np > 0.5, np.nan, 0.6)
    ax.pcolormesh(l_arcsec, m_arcsec, excluded.T, cmap="Greys_r", alpha=0.55, vmin=0, vmax=1)
    # White boundary contour marks the region edge
    ax.contour(l_arcsec, m_arcsec, mask_np.T, levels=[0.5], colors="white", linewidths=1.5)
    plt.tight_layout()
    plt.show()

<a id="lm-mode-shapes"></a>
## lm-mode shapes — angular offsets in arcsec / arcmin

When all shape coordinates are expressed in **arcsec** or **arcmin**, `select_mask`
automatically detects *lm mode* and compares the CRTF distances directly against the
physical `l` and `m` coordinates on the DataArray.  No `coordsys=` keyword is needed.

This is the most natural way to specify regions on a radio-astronomy image: you say
"give me a circle of radius 15 arcsec centred on the image reference direction" and the
code works out which pixels fall inside it.

All center / vertex coordinates are **angular offsets from the reference direction**
(i.e., from l = m = 0).  Positive l is east; positive m is north.

Supported shapes: `circle`, `annulus`, `box`, `centerbox`, `rotbox`, `ellipse`, `poly`.
Each is shown with a short example below.

### circle

Syntax: `circle[[center_l, center_m], radius]`

Selects all pixels within `radius` of the given center.  Here we pick a 15-arcsec
circle centred exactly on the reference direction (l = m = 0), which neatly captures
the Gaussian blob.

In [ ]:
crtf_circle = """
#CRTF
circle[[0arcsec, 0arcsec], 15arcsec]
""".strip()

mask_circle = select_mask(sky, crtf_circle)
print("Selected pixels:", int(mask_circle.values.sum()))
plot_sky_with_mask(sky, mask_circle, title="circle r=15arcsec")

### annulus

Syntax: `annulus[[center_l, center_m], [inner_radius, outer_radius]]`

Selects the ring of pixels whose distance from the center falls between the two radii.
Useful for selecting a background annulus around a source.

In [ ]:
crtf_annulus = """
#CRTF
annulus[[0arcsec, 0arcsec], [20arcsec, 35arcsec]]
""".strip()

mask_annulus = select_mask(sky, crtf_annulus)
print("Selected pixels:", int(mask_annulus.values.sum()))
plot_sky_with_mask(sky, mask_annulus, title="annulus 20–35arcsec")

### centerbox

Syntax: `centerbox[[center_l, center_m], [width, height]]`

Selects a rectangle of `width` × `height` centred at the given position.
The width is along l (east) and the height is along m (north).

Here we select a 30 × 20 arcsec box offset 5 arcsec east and 5 arcsec north of the
reference direction.

In [ ]:
crtf_centerbox = """
#CRTF
centerbox[[5arcsec, 5arcsec], [30arcsec, 20arcsec]]
""".strip()

mask_centerbox = select_mask(sky, crtf_centerbox)
print("Selected pixels:", int(mask_centerbox.values.sum()))
plot_sky_with_mask(sky, mask_centerbox, title="centerbox 30×20arcsec, offset 5arcsec NE")

### box

Syntax: `box[[blc_l, blc_m], [trc_l, trc_m]]`

Selects a rectangle given its bottom-left corner (blc) and top-right corner (trc).
Both corners are angular offsets from the reference direction.

This selects the lower-left quadrant of the image (l < 0, m < 0).

In [ ]:
crtf_box = """
#CRTF
box[[-50arcsec, -50arcsec], [0arcsec, 0arcsec]]
""".strip()

mask_box = select_mask(sky, crtf_box)
print("Selected pixels:", int(mask_box.values.sum()))
plot_sky_with_mask(sky, mask_box, title="box: lower-left quadrant")

### rotbox — rotated rectangle

Syntax: `rotbox[[center_l, center_m], [width, height], pa=<angle>]`

Like `centerbox` but rotated by a position angle.  The angle is specified with one of
two keywords to avoid ambiguity:

- **`pa=<angle>`** — position angle measured from +m (north) toward +l (east).  This is
  the conventional astronomical position angle.
- **`theta_m=<angle>`** — math angle measured from +l (east) toward +m (north).

Angle units: `deg` or `rad`.

Here we select a 40 × 15 arcsec box tilted 30° east of north (pa=30deg).

In [ ]:
crtf_rotbox = """
#CRTF
rotbox[[0arcsec, 0arcsec], [40arcsec, 15arcsec], pa=30deg]
""".strip()

mask_rotbox = select_mask(sky, crtf_rotbox)
print("Selected pixels:", int(mask_rotbox.values.sum()))
plot_sky_with_mask(sky, mask_rotbox, title="rotbox 40×15arcsec, pa=30deg")

### ellipse

Syntax: `ellipse[[center_l, center_m], [semi_major, semi_minor], pa=<angle>]`

The two sizes are the **semi-axes** (half-widths), not full widths.  The position angle
convention (`pa=` or `theta_m=`) is the same as for `rotbox`.

Here the semi-major axis is 20 arcsec along the east–west direction (pa=90deg places
the major axis pointing east).

In [ ]:
crtf_ellipse = """
#CRTF
ellipse[[0arcsec, 0arcsec], [20arcsec, 10arcsec], pa=90deg]
""".strip()

mask_ellipse = select_mask(sky, crtf_ellipse)
print("Selected pixels:", int(mask_ellipse.values.sum()))
plot_sky_with_mask(sky, mask_ellipse, title="ellipse 20×10arcsec semi-axes, pa=90deg")

### poly — arbitrary polygon

Syntax: `poly[[l0, m0], [l1, m1], ..., [lN, mN]]`

Selects pixels inside a polygon defined by a list of vertices (angular offsets).
The polygon is automatically closed (the last vertex connects back to the first).

Here we draw a rough L-shape in the upper portion of the image.

In [ ]:
crtf_poly = (
    "#CRTF\n"
    "poly[[-30arcsec, 10arcsec], [0arcsec, 10arcsec], [0arcsec, 30arcsec],"
    " [20arcsec, 30arcsec], [20arcsec, 40arcsec], [-30arcsec, 40arcsec]]"
)

mask_poly = select_mask(sky, crtf_poly)
print("Selected pixels:", int(mask_poly.values.sum()))
plot_sky_with_mask(sky, mask_poly, title="poly — L-shape")

### arcmin units

All of the examples above use arcsec.  Arcmin works exactly the same way — mix and
match units within a single shape is not allowed, but different shapes in the same CRTF
string may use different units.

Here is a 0.5-arcmin (= 30 arcsec) radius circle to confirm the unit conversion:

In [ ]:
crtf_arcmin = "#CRTF\ncircle[[0arcmin, 0arcmin], 0.5arcmin]"
mask_arcmin = select_mask(sky, crtf_arcmin)

crtf_arcsec = "#CRTF\ncircle[[0arcsec, 0arcsec], 30arcsec]"
mask_arcsec = select_mask(sky, crtf_arcsec)

# The two masks must be identical
np.testing.assert_array_equal(mask_arcmin.values, mask_arcsec.values)
print("0.5arcmin and 30arcsec produce identical masks \u2713")
plot_sky_with_mask(sky, mask_arcmin, title="0.5arcmin = 30arcsec circle")

### Combining lm-mode shapes

Multi-line CRTF works the same as in the pixel-mode notebook.  A leading `+` (or no
prefix) adds the region; a leading `-` subtracts it from the accumulated mask.

Example: select the Gaussian blob core, then punch out an offset circle.

In [ ]:
crtf_combined = """
#CRTF
+circle[[0arcsec, 0arcsec], 25arcsec]
-circle[[5arcsec, -5arcsec], 8arcsec]
""".strip()

mask_combined = select_mask(sky, crtf_combined)
print("Selected pixels:", int(mask_combined.values.sum()))
plot_sky_with_mask(sky, mask_combined, title="25arcsec circle minus 8arcsec core")